# GraphRAG End-to-End Example: Lupus Research

This notebook demonstrates an end-to-end GraphRAG workflow using the [GraphRAG Python package](https://neo4j.com/docs/neo4j-graphrag-python/current/index.html) for Neo4j and [GLiNER2](https://github.com/fastino/gliner2) for knowledge graph extraction.

Starting from unstructured PDF documents, we:
1. **Chunk and embed** PDFs using `SimpleKGPipeline` (creates `Chunk` and `Document` nodes with vector embeddings)
2. **Extract entities and relationships** using GLiNER2 with schema-enforced directions and synonym-based deduplication
3. **Retrieve** knowledge using vector search combined with graph traversal
4. **Answer questions** using GraphRAG (retrieval-augmented generation grounded in the knowledge graph)

**Prerequisites:**
- A running Neo4j instance with the [APOC plugin](https://neo4j.com/docs/apoc/current/installation/) installed
- An OpenAI API key (for embeddings and RAG answer generation — entity extraction uses GLiNER2 locally)
- A `.env` file with `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`, and `OPENAI_API_KEY`

## Setup

In [1]:
%%capture
%pip install neo4j-graphrag[openai] neo4j pypdf python-dotenv tiktoken gliner2 torch pyyaml

In [2]:
from dotenv import load_dotenv
import os

load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')

In [3]:
import neo4j
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings

driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD), database=NEO4J_DATABASE)
embedder = OpenAIEmbeddings()

## Phase 1: PDF Chunking and Embedding

We use `SimpleKGPipeline` to parse PDFs, split text into chunks, and create vector embeddings.
Entity and relationship lists are left empty — `SimpleKGPipeline` handles only the lexical graph
(Chunk and Document nodes). Entity extraction is done separately by GLiNER2 in Phase 2.

In [4]:
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.experimental.components.text_splitters.fixed_size_splitter import FixedSizeSplitter
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

kg_builder_pdf = SimpleKGPipeline(
    llm=OpenAILLM(model_name='gpt-4o-mini', model_params={'response_format': {'type': 'json_object'}, 'temperature': 0}),
    driver=driver,
    text_splitter=FixedSizeSplitter(chunk_size=500, chunk_overlap=100),
    embedder=embedder,
    entities=[],
    relations=[],
    from_pdf=True
)

In [5]:
pdf_file_paths = [
    'truncated-pdfs/biomolecules-11-00928-v2-trunc.pdf',
    'truncated-pdfs/GAP-between-patients-and-clinicians_2023_Best-Practice-trunc.pdf',
    'truncated-pdfs/pgpm-13-39-trunc.pdf'
]

for path in pdf_file_paths:
    print(f'Processing: {path}')
    pdf_result = await kg_builder_pdf.run_async(file_path=path)
    print(f'Result: {pdf_result}')

Processing: truncated-pdfs/biomolecules-11-00928-v2-trunc.pdf


Result: run_id='032803da-99ac-4a3b-9f3d-053910b4682d' result={'resolver': {'number_of_nodes_to_resolve': 364, 'number_of_created_nodes': 61}}
Processing: truncated-pdfs/GAP-between-patients-and-clinicians_2023_Best-Practice-trunc.pdf


Result: run_id='bd7cde3e-87f2-45c7-aa3c-a5cc98a91b0a' result={'resolver': {'number_of_nodes_to_resolve': 556, 'number_of_created_nodes': 107}}
Processing: truncated-pdfs/pgpm-13-39-trunc.pdf


Result: run_id='b4b8fc17-fbee-4ca7-ae06-e81f7393f495' result={'resolver': {'number_of_nodes_to_resolve': 548, 'number_of_created_nodes': 107}}


In [6]:
# SimpleKGPipeline may create some entity nodes even with empty lists.
# Clean those, keeping only Chunk and Document nodes.
with driver.session() as session:
    result = session.run(
        'MATCH (n:__Entity__) WHERE NOT n:Chunk AND NOT n:Document DETACH DELETE n '
        'RETURN count(*) AS deleted'
    )
    print(f'Cleaned up {result.single()["deleted"]} LLM artifact nodes')

Cleaned up 548 LLM artifact nodes


## Phase 2: Knowledge Graph Extraction with GLiNER2

GLiNER2 extracts entities and relationships from the text chunks created in Phase 1.
The extraction is configured via `configs/lupus_kg.yaml`, which defines:
- **Entity types**: Disease, Drug, GeneOrProtein, Biomarker, Symptom, CellType, Pathway, Anatomy
- **Relationship types**: TREATS (Drug→Disease), BIOMARKER_FOR (Biomarker→Disease), CAUSES, ASSOCIATED_WITH, etc.
- **Synonym table**: Maps variants (SLE, lupus) to canonical names (Systemic Lupus Erythematosus)

A `RELATION_SCHEMA` enforces correct relationship directions — it is structurally impossible to
create `Disease → TREATS → Drug` (the reversed direction that plagues LLM-based extraction).

In [7]:
from lupus_kg.extraction import run as run_kg_extraction

stats = run_kg_extraction(config_path='configs/lupus_kg.yaml')
print(f'Entities created: {stats["nodes_created"]}')
print(f'Relationships created: {stats["relationships_created"]}')
print(f'Errors: {stats["errors"]}')

/Users/alexwoolford/anaconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


Entities created: 1163
Relationships created: 81
Errors: 0


## Knowledge Graph Retrieval

We create a vector index on the text chunks for similarity search.

In [8]:
from neo4j_graphrag.indexes import create_vector_index

create_vector_index(driver, name='text_embeddings', label='Chunk',
                    embedding_property='embedding', dimensions=1536, similarity_fn='cosine')

The __VectorRetriever__ queries `Chunk` nodes via vector search, returning text and metadata.

In [9]:
from neo4j_graphrag.retrievers import VectorRetriever

vector_retriever = VectorRetriever(
    driver,
    index_name='text_embeddings',
    embedder=embedder,
    return_properties=['text'],
)

Below we visualize the context from a vector search.

In [10]:
import json

vector_res = vector_retriever.get_search_results(
    query_text='How is precision medicine applied to Lupus?', top_k=3
)
for i in vector_res.records:
    print('====\n' + json.dumps(i.data(), indent=4))

====
{
    "node": {
        "text": "therapy \u22647.5 mg of predni-\nsone and well-tolerated IS agents.\nPrecision medicine consists of a tailored approach to\neach patient, based on genetic and epigenetic singularities,\nwhich in \ufb02uence disease pathophysiology and drug\nresponse. Precision medicine in SLE is trying to address\nthe need to assess SLE patients optimally, predict disease\ncourse and treatment response at diagnosis. Ideally every\npatient would undergo an initial evaluation that would\npro\ufb01le his/her disease, assessing the main "
    },
    "nodeLabels": [
        "__KGBuilder__",
        "Chunk"
    ],
    "elementId": "4:efac5a06-0be1-40c4-b9a4-b59d59d1ea65:507",
    "id": "4:efac5a06-0be1-40c4-b9a4-b59d59d1ea65:507",
    "score": 0.9471423625946045
}
====
{
    "node": {
        "text": " will be susceptible to a precision medicine approach.\nConclusions\nWe have analysed recent facets of the pathological and\nimmunogenetic basis of lupus. The network of fa

The __`VectorCypherRetriever`__ combines vector search with graph traversal. It finds relevant
text chunks, then follows `FROM_CHUNK` edges to extracted entities and traverses their relationships
1-2 hops out. This provides the LLM with both raw text and structured knowledge graph context.

In [11]:
from neo4j_graphrag.retrievers import VectorCypherRetriever

vc_retriever = VectorCypherRetriever(
    driver,
    index_name='text_embeddings',
    embedder=embedder,
    retrieval_query="""
WITH node AS chunk
MATCH (chunk)<-[:FROM_CHUNK]-()-[relList:!FROM_CHUNK]-{1,2}()
UNWIND relList AS rel
WITH collect(DISTINCT chunk) AS chunks,
  collect(DISTINCT rel) AS rels
RETURN '=== text ===' + '\n' + apoc.text.join([c in chunks | c.text], '\n---\n') +
  '\n\n=== kg_rels ===' + '\n' +
  apoc.text.join([r in rels | startNode(r).name + ' - ' + type(r) +
  '(' + coalesce(r.evidence_text, '') + ')' + ' -> ' + endNode(r).name], '\n---\n') AS info
"""
)

Below we visualize the context from the hybrid retriever — both text chunks and KG relationships.

In [12]:
vc_res = vc_retriever.get_search_results(
    query_text='How is precision medicine applied to Lupus?', top_k=3
)

kg_rel_pos = vc_res.records[0]['info'].find('\n\n=== kg_rels ===')
print('# Text Chunk Context:')
print(vc_res.records[0]['info'][:kg_rel_pos])
print('# KG Context From Relationships:')
print(vc_res.records[0]['info'][kg_rel_pos:])

# Text Chunk Context:
=== text ===
therapy ≤7.5 mg of predni-
sone and well-tolerated IS agents.
Precision medicine consists of a tailored approach to
each patient, based on genetic and epigenetic singularities,
which in ﬂuence disease pathophysiology and drug
response. Precision medicine in SLE is trying to address
the need to assess SLE patients optimally, predict disease
course and treatment response at diagnosis. Ideally every
patient would undergo an initial evaluation that would
proﬁle his/her disease, assessing the main 
---
 will be susceptible to a precision medicine approach.
Conclusions
We have analysed recent facets of the pathological and
immunogenetic basis of lupus. The network of factors that
drives organ selectivity and how the precise within indivi-
duals conspires to cause the diverse clinical features
remains a mystery. Current treatment can be approached
in a more precise and systematic fashion as suggested here.
Future care will involve molecular diagnostics throu

## GraphRAG

The `GraphRAG` class combines a retriever with an LLM to answer questions grounded in the
knowledge graph context. We use GPT-4o for answer generation with a template that constrains
the LLM to only use information from the retrieved context.

In [13]:
from neo4j_graphrag.llm import OpenAILLM as LLM
from neo4j_graphrag.generation import RagTemplate
from neo4j_graphrag.generation.graphrag import GraphRAG

llm = LLM(model_name='gpt-4o', model_params={'temperature': 0.0})

rag_template = RagTemplate(template='''Answer the Question using the following Context. Only respond with information mentioned in the Context. Do not inject any speculative information not mentioned.

# Question:
{query_text}

# Context:
{context}

# Answer:
''', expected_inputs=['query_text', 'context'])

v_rag  = GraphRAG(llm=llm, retriever=vector_retriever, prompt_template=rag_template)
vc_rag = GraphRAG(llm=llm, retriever=vc_retriever, prompt_template=rag_template)

Now we run GraphRAG and compare Vector-only vs Vector+Cypher responses.

In [14]:
q = 'How is precision medicine applied to Lupus? provide in list format.'
print(f'Vector Response: \n{v_rag.search(q, retriever_config={"top_k":5}).answer}')
print('\n===========================\n')
print(f'Vector + Cypher Response: \n{vc_rag.search(q, retriever_config={"top_k":5}).answer}')

Vector Response: 
- Precision medicine in lupus involves a tailored approach to each patient based on genetic and epigenetic singularities.
- It aims to influence disease pathophysiology and drug response.
- Precision medicine seeks to optimally assess SLE patients, predict disease course, and treatment response at diagnosis.
- Ideally, every patient would undergo an initial evaluation to profile their disease.




Vector + Cypher Response: 
- Precision medicine in lupus involves a tailored approach to each patient based on genetic and epigenetic singularities.
- It aims to optimally assess SLE patients, predict disease course, and treatment response at diagnosis.
- Ideally, every patient would undergo an initial evaluation to profile their disease, assessing the main factors influencing pathophysiology and drug response.


In [15]:
q = 'Can you summarize systemic lupus erythematosus (SLE)? including common effects, biomarkers, and treatments? Provide in detailed list format.'
print(f'Vector Response: \n{v_rag.search(q, retriever_config={"top_k": 5}).answer}')
print('\n===========================\n')
print(f'Vector + Cypher Response: \n{vc_rag.search(q, retriever_config={"top_k": 5}).answer}')

Vector Response: 
- **Common Effects of Systemic Lupus Erythematosus (SLE):**
  - SLE is a systemic autoimmune disease characterized by aberrant activity of the immune system.
  - It presents with a wide range of clinical features and immunological abnormalities.
  - The disease involves multiorgan involvement and has a highly variable course with considerable inter-individual variability.

- **Biomarkers for SLE:**
  - Biomarkers are used to diagnose and monitor disease activity in SLE, with and without organ-specific injury.
  - Common biomarkers reflect immune reactivity and inflammation in various organs.
  - Ideal biomarkers should reflect the underlying pathophysiology or treatment target, have reliability, validity, high predictive values, and high sensitivity.
  - Novel biomarkers have been discovered through “omics” research.

- **Treatments for SLE:**
  - Common treatments include the use of steroids, immunosuppressives, and hydroxychloroquine.
  - These treatments are broadl

Vector + Cypher Response: 
Systemic Lupus Erythematosus (SLE) is a chronic, complex autoimmune disease with multiorgan involvement and a highly variable course. Here is a detailed summary based on the provided context:

### Common Effects:
- **Multiple Organ Damage**: SLE can cause damage to various organs, leading to a wide range of clinical symptoms and signs.
- **Immunological Abnormalities**: The disease is characterized by aberrant activity of the immune system.

### Common Biomarkers:
- **Anti-dsDNA Antibodies**: Associated with disease activity and renal disease.
- **Anti-Sm Antibodies**: Highly specific for SLE, correlated with disease activity, and associated with lupus nephritis (LN).
- **Complement Proteins (C3, C4)**: Used in diagnosing SLE and assessing disease activity.
- **Antiphospholipid Antibodies**: Used in the classification criteria for SLE.
- **Transcriptome and Epigenetic Markers**: Used to track disease course and understand pathogenesis.
- **Metabolites (L-vali

In [16]:
q = 'Can you summarize systemic lupus erythematosus (SLE)? including common effects, biomarkers, treatments, and current challenges faced by Physicians and patients? provide in list format with details for each item.'
print(f'Vector Response: \n{v_rag.search(q, retriever_config={"top_k": 5}).answer}')
print('\n===========================\n')
print(f'Vector + Cypher Response: \n{vc_rag.search(q, retriever_config={"top_k": 5}).answer}')

Vector Response: 
- **Common Effects:**
  - Systemic lupus erythematosus (SLE) is characterized by a wide range of clinical features and immunological abnormalities.
  - It is a chronic inflammatory systemic disease with multi-organ involvement and a complex clinical picture.
  - The disease presents with varying severity and an unpredictable relapsing and remitting course.

- **Biomarkers:**
  - Biomarkers are used to diagnose and monitor disease activity in SLE, with and without organ-specific injury.
  - Common biomarkers should reflect the underlying pathophysiology or treatment target and have reliability, validity, high predictive values, and high sensitivity.
  - Novel SLE biomarkers have been discovered through “omics” research.

- **Treatments:**
  - Current treatments include broadly similar types of drugs such as steroids, immunosuppressives, and hydroxychloroquine.

- **Current Challenges:**
  - Finding an ideal biomarker for SLE is challenging due to the need for it to mee

Vector + Cypher Response: 
- **Common Effects of SLE:**
  - SLE is a systemic autoimmune disease characterized by aberrant activity of the immune system.
  - It presents with a wide range of clinical features and immunological abnormalities.
  - SLE can cause multiple organ damage and has a complex clinical picture with varying severity and unpredictable relapsing.

- **Biomarkers for SLE:**
  - Common biomarkers include anti-dsDNA antibodies, anti-Sm antibodies, complement proteins, and antiphospholipid antibodies.
  - Novel biomarkers discovered through "omics" research include transcriptome, epigenetic markers, and specific metabolites like L-valine, pyrimidine, erucamide, and L-leucine.
  - Finding an ideal biomarker is challenging due to the need for high sensitivity, specificity, and reflection of the underlying pathophysiology.

- **Treatments for SLE:**
  - Common treatments include steroids, immunosuppressives, hydroxychloroquine, and antimalarials.
  - Anifrolumab is a newer 